# 🚀 MINav Offline RL Training on Kaggle (Simulation Exploration Dataset)
### End-to-End Pipeline: Simulation Exploration Data → DINOv3 (ViT-S) Visual Embeddings → Geometric & Uniform Hindsight Relabeling → TD3+BC Offline RL

This notebook processes the uploaded **1-Hour Simulation Exploration Dataset** (`simulation-run`, 36 segments, 36,000 steps, 72,000 RGB/depth frames) collected with a 10Hz Pink Noise policy in the 16x16m indoor complex navigation maze, and trains an offline goal-conditioned navigation policy using **TD3+BC**.

#### ⚙️ Recommended Kaggle Settings:
- **Accelerator**: **GPU T4 x2** or **GPU P100**
- **Internet**: **ON** (required for `git clone` & downloading DINOv3 pretrained weights via `timm`)
- **Persistence**: Files in `/kaggle/working/` persist during the active session


In [ ]:
# Cell 1: Environment & GPU Verification
import os, sys, subprocess, torch

print('=' * 60)
print('SYSTEM & GPU STATUS')
print('=' * 60)
print(f'PyTorch Version : {torch.__version__}')
print(f'CUDA Available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device Name     : {torch.cuda.get_device_name(0)}')
    print(f'Device Count    : {torch.cuda.device_count()}')
    print(f'VRAM Total      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('⚠️ WARNING: GPU not detected! Make sure to select GPU T4 x2 or P100 in Kaggle Settings.')
print('=' * 60)

# Install required dependencies
!pip install -q pyarrow pandas pyyaml wandb tensorboard timm torchvision matplotlib
print('✓ Dependencies successfully installed.')


In [ ]:
# Cell 2: Clone / Update Repository
REPO_DIR = '/kaggle/working/RL-Image-Goal-Navigation'

if not os.path.exists(REPO_DIR):
    print(f'Cloning repository into {REPO_DIR}...')
    !git clone https://github.com/sanmita13742/RL-Image-Goal-Navigation.git {REPO_DIR}
else:
    print('Repository exists. Fetching latest updates from origin/main...')
    !git -C {REPO_DIR} reset --hard origin/main
    !git -C {REPO_DIR} pull origin main

%cd {REPO_DIR}
SIM_DIR = os.path.join(REPO_DIR, 'simulation', 'ranger_mini_v3')
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
if SIM_DIR not in sys.path:
    sys.path.insert(0, SIM_DIR)
print(f'✓ Active Working Directory: {os.getcwd()}')


In [ ]:
# Cell 3: Auto-Detect Uploaded Kaggle Dataset (Zip or Folder)
import os, sys, shutil, zipfile
from pathlib import Path

print('=' * 60)
print('SEARCHING FOR SIMULATION EXPLORATION DATASET IN /kaggle/input')
print('=' * 60)

input_dir = Path('/kaggle/input')
explore_dir = None

# 1. First check if dataset is already extracted in /kaggle/input
for p in input_dir.rglob('exploration_metadata.json'):
    explore_dir = p.parent
    print(f'✓ Found exploration metadata at: {p}')
    break

if explore_dir is None:
    for p in input_dir.rglob('segment_000'):
        explore_dir = p.parent
        print(f'✓ Found segment_000 at: {p}')
        break

# 2. If not extracted, look for a .zip archive in /kaggle/input
if explore_dir is None:
    zip_candidates = list(input_dir.rglob('*.zip'))
    if zip_candidates:
        target_zip = zip_candidates[0]
        print(f'✓ Found archive: {target_zip}. Extracting to /kaggle/working/extracted_dataset...')
        extract_dir = Path('/kaggle/working/extracted_dataset')
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(target_zip, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)
        
        # Search inside extracted directory
        for p in extract_dir.rglob('exploration_metadata.json'):
            explore_dir = p.parent
            break
        if explore_dir is None:
            for p in extract_dir.rglob('segment_000'):
                explore_dir = p.parent
                break

if explore_dir is None:
    print('❌ Exploration dataset not found in /kaggle/input.')
    print('Items found in /kaggle/input:')
    for item in input_dir.rglob('*'):
        if item.is_dir() or item.suffix in ('.zip', '.json', '.csv'):
            print(' -', item)
    raise FileNotFoundError('Please attach your simulation dataset (simulation-run or simulation-run.zip) to the Kaggle notebook.')

print(f'\nTarget Exploration Directory: {explore_dir}')

# Set up execution run directory
RUN_NAME = 'simulation_run'
RUN_DIR = Path(f'/kaggle/working/runs/{RUN_NAME}')
RUN_DIR.mkdir(parents=True, exist_ok=True)
EXPLORATION_LINK = RUN_DIR / 'exploration'

# Link or copy exploration directory
if EXPLORATION_LINK.exists():
    if EXPLORATION_LINK.is_symlink():
        EXPLORATION_LINK.unlink()
    else:
        shutil.rmtree(EXPLORATION_LINK)

try:
    os.symlink(str(explore_dir), str(EXPLORATION_LINK))
    print(f'✓ Symlinked: {explore_dir} -> {EXPLORATION_LINK}')
except Exception:
    shutil.copytree(str(explore_dir), str(EXPLORATION_LINK))
    print(f'✓ Copied: {explore_dir} -> {EXPLORATION_LINK}')

segments = sorted(EXPLORATION_LINK.glob('segment_*'))
print(f'✓ Total exploration segments detected: {len(segments)}')
assert len(segments) > 0, 'No segments found in exploration directory!'


In [ ]:
# Cell 4: Configure Simulation Run Parameters
import os, sys, yaml, torch
from pathlib import Path

REPO_DIR = Path('/kaggle/working/RL-Image-Goal-Navigation')
SIM_DIR = REPO_DIR / 'simulation' / 'ranger_mini_v3'
RUN_DIR = Path('/kaggle/working/runs/simulation_run')

# Load base pipeline configuration
base_config_file = SIM_DIR / 'configs' / 'pipeline.yaml'
with open(base_config_file, 'r') as f:
    config = yaml.safe_load(f)

# Optimal Hyperparameters from Empirical Audit
TRAIN_STEPS      = 50_000        # Total gradient steps
BATCH_SIZE       = 1024          # Mini-batch size (512 geo + 512 unif)
LR_ACTOR         = 1.0e-4        # Stable actor learning rate
LR_CRITIC        = 1.0e-4        # Stable critic learning rate
RANDOM_SEED      = 42            # Reproducibility seed
LAMBDA_BC        = 0.001         # Behavior cloning regularization weight
GAMMA            = 0.99          # Discount factor
TAU              = 0.005         # Soft target update rate
POLICY_DELAY     = 2             # Delayed policy updates (TD3)
TARGET_NOISE     = 0.2           # Target action smoothing noise
TARGET_NOISE_CLIP= 0.5           # Target noise clip
FQE_EVAL_EVERY   = 1000          # Fitted Q-Evaluation frequency
CHECKPOINT_EVERY = 10_000        # Checkpoint saving frequency
DEVICE           = 'cuda' if torch.cuda.is_available() else 'cpu'

if 'run' not in config:
    config['run'] = {}
config['run']['seed'] = RANDOM_SEED
config['run']['device'] = DEVICE

if 'training' not in config:
    config['training'] = {}
config['training']['total_gradient_steps'] = TRAIN_STEPS
config['training']['batch_size'] = BATCH_SIZE
config['training']['lr_actor'] = LR_ACTOR
config['training']['lr_critic'] = LR_CRITIC
config['training']['lambda_bc'] = LAMBDA_BC
config['training']['gamma'] = GAMMA
config['training']['tau'] = TAU
config['training']['policy_delay'] = POLICY_DELAY
config['training']['target_noise'] = TARGET_NOISE
config['training']['target_noise_clip'] = TARGET_NOISE_CLIP
config['training']['fqe_eval_every'] = FQE_EVAL_EVERY
config['training']['fqe_frequency'] = FQE_EVAL_EVERY
config['training']['checkpoint_every'] = CHECKPOINT_EVERY
config['training']['action_scale'] = [1.8, 1.0, 1.5]

# Write both config_run.yaml and config_used.yaml
CONFIG_RUN = RUN_DIR / 'config_run.yaml'
with open(CONFIG_RUN, 'w') as f:
    yaml.dump(config, f)
with open(RUN_DIR / 'config_used.yaml', 'w') as f:
    yaml.dump(config, f)

print('=' * 60)
print(f'✓ Configuration written to: {CONFIG_RUN}')
print('=' * 60)
print(f'  - Accelerator        : {DEVICE}')
print(f'  - Random Seed        : {RANDOM_SEED}')
print(f'  - Total Steps        : {TRAIN_STEPS:,}')
print(f'  - Batch Size         : {BATCH_SIZE:,}')
print(f'  - Actor LR           : {LR_ACTOR}')
print(f'  - Critic LR          : {LR_CRITIC}')
print(f'  - BC Weight (Lambda) : {LAMBDA_BC}')
print(f'  - Discount (Gamma)   : {GAMMA}')
print(f'  - Polyak Tau         : {TAU}')
print(f'  - Policy Delay       : {POLICY_DELAY}')
print(f'  - Target Policy Noise: {TARGET_NOISE} (clip: {TARGET_NOISE_CLIP})')
print(f'  - FQE Eval Interval  : {FQE_EVAL_EVERY:,}')
print(f'  - Checkpoint Interval: {CHECKPOINT_EVERY:,}')
print('=' * 60)


In [ ]:
# Cell 5: Phase 2 — DINOv3 (ViT-Small) Encoding & Hindsight Dataset Generation
import os, sys, subprocess, shutil, torch
from pathlib import Path

REPO_DIR = Path('/kaggle/working/RL-Image-Goal-Navigation')
SIM_DIR = REPO_DIR / 'simulation' / 'ranger_mini_v3'
RUN_DIR = Path('/kaggle/working/runs/simulation_run')
CONFIG_RUN = RUN_DIR / 'config_run.yaml'

print('=' * 60)
print('PHASE 2: DINOv3 VISUAL ENCODING & HINDSIGHT GOAL GENERATION')
print('=' * 60)

hindsight_script = SIM_DIR / 'scripts' / 'build_final_dataset.py'
cmd = [
    sys.executable, str(hindsight_script),
    '--config', str(CONFIG_RUN),
    '--run-dir', str(RUN_DIR)
]

print('Executing command:', ' '.join(cmd))
subprocess.run(cmd, check=True)

# Guarantee cross-module compatibility for phi vectors
dinov3_dir = RUN_DIR / 'processed' / 'dinov3'
if (dinov3_dir / 'phi_cache.npy').exists() and not (dinov3_dir / 'phi_vectors.npy').exists():
    shutil.copy(dinov3_dir / 'phi_cache.npy', dinov3_dir / 'phi_vectors.npy')
    print('✓ Created phi_vectors.npy alias from phi_cache.npy')
elif (dinov3_dir / 'phi_vectors.npy').exists() and not (dinov3_dir / 'phi_cache.npy').exists():
    shutil.copy(dinov3_dir / 'phi_vectors.npy', dinov3_dir / 'phi_cache.npy')
    print('✓ Created phi_cache.npy alias from phi_vectors.npy')

print('\n✓ Phase 2 Hindsight Dataset Generation successfully completed!')


In [ ]:
# Cell 6: Verify Processed DINOv3 Embeddings & Hindsight Parquet Transitions
import numpy as np
import pandas as pd
from pathlib import Path

RUN_DIR = Path('/kaggle/working/runs/simulation_run')
processed_dir = RUN_DIR / 'processed'
dinov3_dir = processed_dir / 'dinov3'
hindsight_dir = processed_dir / 'hindsight'

print('=' * 60)
print('DATASET AUDIT & SHAPE VERIFICATION')
print('=' * 60)

phi_path = dinov3_dir / 'phi_vectors.npy' if (dinov3_dir / 'phi_vectors.npy').exists() else dinov3_dir / 'phi_cache.npy'
if phi_path.exists():
    phi = np.load(phi_path, mmap_mode='r')
    l2_norm = float(np.linalg.norm(phi[0]))
    print(f'✓ DINOv3 Embeddings Shape : {phi.shape} (Expected: [N, 384])')
    print(f'  L2 Unit Norm Check      : {l2_norm:.4f} (Expected: 1.0000)')
    assert abs(l2_norm - 1.0) < 1e-3, f'L2 norm check failed: {l2_norm}'

ssd_path = dinov3_dir / 'ssd_scores.npy'
if ssd_path.exists():
    ssd = np.load(ssd_path)
    print(f'✓ SSD Stability Scores    : {len(ssd):,} frames (Mean: {ssd.mean():.4f}, Min: {ssd.min():.4f}, Max: {ssd.max():.4f})')

geom_path = hindsight_dir / 'geometric_transitions.parquet'
if geom_path.exists():
    df_geom = pd.read_parquet(geom_path)
    print(f'✓ Geometric Transitions   : {len(df_geom):,} transitions (r=1 rate: {100 * df_geom["reward"].mean():.2f}%)')

unif_path = hindsight_dir / 'uniform_transitions.parquet'
if unif_path.exists():
    df_unif = pd.read_parquet(unif_path)
    print(f'✓ Uniform Transitions     : {len(df_unif):,} transitions (r=1 rate: {100 * df_unif["reward"].mean():.2f}%)')

report_file = processed_dir / 'FINAL_HINDSIGHT_DATASET_REPORT.md'
if report_file.exists():
    print(f'✓ Validation Report Found : {report_file}')
print('=' * 60)


In [ ]:
# Cell 7: Phase 3 — Train Offline TD3+BC Navigation Policy
import os, sys, subprocess, torch
from pathlib import Path

REPO_DIR = Path('/kaggle/working/RL-Image-Goal-Navigation')
SIM_DIR = REPO_DIR / 'simulation' / 'ranger_mini_v3'
RUN_DIR = Path('/kaggle/working/runs/simulation_run')
CONFIG_RUN = RUN_DIR / 'config_run.yaml'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('=' * 60)
print('PHASE 3: OFFLINE TD3+BC ACTOR-CRITIC TRAINING')
print('=' * 60)

train_script = SIM_DIR / 'scripts' / 'train_td3_bc.py'
cmd = [
    sys.executable, str(train_script),
    '--config', str(CONFIG_RUN),
    '--run-dir', str(RUN_DIR),
    '--device', DEVICE
]

print('Executing command:', ' '.join(cmd))
subprocess.run(cmd, check=True)
print('\n✓ Phase 3 TD3+BC Offline Training completed successfully!')


In [ ]:
# Cell 8: Training Metrics Visualization Dashboard (Standard Clean White Background)
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

RUN_DIR = Path('/kaggle/working/runs/simulation_run')
metrics_csv = RUN_DIR / 'training' / 'metrics.csv'

if not metrics_csv.exists():
    print(f'Metrics file {metrics_csv} not found yet.')
else:
    df = pd.read_csv(metrics_csv)
    print(f'Loaded {len(df):,} metric records up to step {df["step"].max():,}.')
    
    # Standard white background styling
    plt.style.use('default')
    fig, axes = plt.subplots(2, 3, figsize=(18, 10), facecolor='white')
    
    # 1. Critic Loss
    ax = axes[0, 0]
    ax.plot(df['step'], df['critic_loss'], color='#1f77b4', lw=1.5, label='Critic Loss')
    ax.set_title('Critic Loss (MSE Bellman Error)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Gradient Steps')
    ax.set_ylabel('Loss')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='upper right')
    
    # 2. Q-Values (Q1, Q2, Target Q)
    ax = axes[0, 1]
    if 'q1_mean' in df.columns:
        ax.plot(df['step'], df['q1_mean'], color='#2ca02c', lw=1.5, label='Q1 Mean')
    if 'q2_mean' in df.columns:
        ax.plot(df['step'], df['q2_mean'], color='#17becf', lw=1.2, linestyle='--', label='Q2 Mean')
    if 'target_q_mean' in df.columns:
        ax.plot(df['step'], df['target_q_mean'], color='#ff7f0e', lw=1.2, linestyle=':', label='Target Q')
    ax.set_title('Critic Q-Value Estimates', fontsize=12, fontweight='bold')
    ax.set_xlabel('Gradient Steps')
    ax.set_ylabel('Estimated Q')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='upper left')
    
    # 3. Actor Loss
    ax = axes[0, 2]
    ax.plot(df['step'], df['actor_loss'], color='#d62728', lw=1.5, label='Actor Loss')
    ax.set_title('Actor Objective Loss', fontsize=12, fontweight='bold')
    ax.set_xlabel('Gradient Steps')
    ax.set_ylabel('Loss')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='upper right')
    
    # 4. Behavior Cloning Loss
    ax = axes[1, 0]
    ax.plot(df['step'], df['bc_loss'], color='#9467bd', lw=1.5, label='BC MSE Loss')
    ax.set_title('Behavior Cloning Loss (||π(s) - a||²)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Gradient Steps')
    ax.set_ylabel('MSE')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='upper right')
    
    # 5. Critic vs BC Loss Ratio
    ax = axes[1, 1]
    ratio = df['critic_loss'] / (df['bc_loss'] + 1e-6)
    ax.plot(df['step'], ratio, color='#8c564b', lw=1.5, label='Critic / BC Ratio')
    ax.set_title('Optimization Dynamics Ratio', fontsize=12, fontweight='bold')
    ax.set_xlabel('Gradient Steps')
    ax.set_ylabel('Ratio')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='upper right')
    
    # 6. Smoothed Q-Value Trajectory
    ax = axes[1, 2]
    if 'q1_mean' in df.columns:
        rolling_q = df['q1_mean'].rolling(window=10, min_periods=1).mean()
        rolling_std = df['q1_mean'].rolling(window=10, min_periods=1).std().fillna(0)
        ax.plot(df['step'], rolling_q, color='#2ca02c', lw=2, label='Q1 (Rolling Mean)')
        ax.fill_between(df['step'], rolling_q - rolling_std, rolling_q + rolling_std, color='#2ca02c', alpha=0.2)
    ax.set_title('Q-Value Convergence & Variance', fontsize=12, fontweight='bold')
    ax.set_xlabel('Gradient Steps')
    ax.set_ylabel('Q-Value')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='upper left')
    
    plt.tight_layout()
    dashboard_path = RUN_DIR / 'training' / 'training_dashboard.png'
    plt.savefig(dashboard_path, dpi=200, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'✓ Publication-quality dashboard saved to: {dashboard_path}')


In [ ]:
# Cell 9: Package Trained Checkpoints & Logs for Download
import os, zipfile
from pathlib import Path

RUN_DIR = Path('/kaggle/working/runs/simulation_run')
train_dir = RUN_DIR / 'training'
OUTPUT_ZIP = Path('/kaggle/working/minav_simulation_policy.zip')

print('=' * 60)
print('PACKAGING SIMULATION POLICY & ARTIFACTS')
print('=' * 60)

with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # 1. Training checkpoints & metrics
    if train_dir.exists():
        for f in train_dir.rglob('*'):
            if f.is_file():
                zipf.write(f, arcname=f'training/{f.relative_to(train_dir)}')
                print(f'  + Included: training/{f.relative_to(train_dir)}')
                
    # 2. Config used
    for cfg in ['config_run.yaml', 'config_used.yaml']:
        cfg_path = RUN_DIR / cfg
        if cfg_path.exists():
            zipf.write(cfg_path, arcname=cfg)
            print(f'  + Included: {cfg}')
            
    # 3. Hindsight report
    report_path = RUN_DIR / 'processed' / 'FINAL_HINDSIGHT_DATASET_REPORT.md'
    if report_path.exists():
        zipf.write(report_path, arcname='FINAL_HINDSIGHT_DATASET_REPORT.md')
        print('  + Included: FINAL_HINDSIGHT_DATASET_REPORT.md')

print('=' * 60)
print(f'✓ Export archive generated successfully!')
print(f'  Path: {OUTPUT_ZIP}')
print(f'  Size: {OUTPUT_ZIP.stat().st_size / 1e6:.2f} MB')
print('You can download "minav_simulation_policy.zip" directly from the Kaggle Output panel.')
print('=' * 60)
